In [1]:
import numpy as np
import time
import random
import copy
import matplotlib.pyplot as plt

In [2]:
class GAPProblem:
    def __init__(self, m, n, profits, resources, capacities):
        self.m = m  
        self.n = n  
        self.profits = np.array(profits)      
        self.resources = np.array(resources)  
        self.capacities = np.array(capacities)
        
        # --- CACHING HASHMAPS ---
        self.repair_cache = {}
        self.eval_cache = {}

    def clear_caches(self):
        """Call this at the start of every algorithm's run() to keep time comparisons fair."""
        self.repair_cache.clear()
        self.eval_cache.clear()

    def repair_assignment(self, assignment):
        key = tuple(assignment)
        if key in self.repair_cache:
            return self.repair_cache[key].copy() # Return copy to prevent accidental overwrites
            
        assignment_copy = np.clip(assignment, 0, self.m - 1).astype(int)
        usage = np.zeros(self.m)
        for j in range(self.n):
            usage[assignment_copy[j]] += self.resources[assignment_copy[j], j]
            
        overloaded = np.where(usage > self.capacities)[0]
        for aOver in overloaded:
            jobsAssigned = np.where(assignment_copy == aOver)[0]
            jobsAssigned = jobsAssigned[np.argsort(-self.resources[aOver, jobsAssigned])]
            
            for j in jobsAssigned:
                if usage[aOver] <= self.capacities[aOver]: break
                
                bestAlt, maxProfitGain = -1, -np.inf
                for aAlt in range(self.m):
                    if aAlt != aOver and (usage[aAlt] + self.resources[aAlt, j] <= self.capacities[aAlt]):
                        gain = self.profits[aAlt, j] - self.profits[aOver, j]
                        if gain > maxProfitGain:
                            maxProfitGain, bestAlt = gain, aAlt
                
                if bestAlt != -1:
                    usage[aOver] -= self.resources[aOver, j]
                    usage[bestAlt] += self.resources[bestAlt, j]
                    assignment_copy[j] = bestAlt
                    
        self.repair_cache[key] = assignment_copy.copy()
        return assignment_copy

    def evaluate(self, assignment):
        key = tuple(assignment)
        if key in self.eval_cache:
            return self.eval_cache[key]
            
        usage = np.zeros(self.m)
        totalProfit = 0
        for j in range(self.n):
            agent = assignment[j]
            totalProfit += self.profits[agent, j]
            usage[agent] += self.resources[agent, j]
            
        violation = sum(max(0, usage[i] - self.capacities[i]) for i in range(self.m))
        penaltyFactor = 10000  
        fitness = totalProfit - (penaltyFactor * violation)
        
        self.eval_cache[key] = fitness
        return fitness

In [ ]:
class BCGA:
    def __init__(self, problem, pop_size=100, gens=500):
        self.p = problem
        self.POP = pop_size
        self.GEN = gens
        self.MUT_MAX = 0.15
        self.MUT_MIN = 0.02
        self.CROSS = 0.8
        self.ELITE_FRAC = 0.1
        # Bits required to represent 'm' agents
        self.bits_per_job = int(np.ceil(np.log2(self.p.m)))
        self.chrom_length = self.p.n * self.bits_per_job

    def decode(self, chrom):
        """Converts Binary chromosome to discrete assignment"""
        assignment = np.zeros(self.p.n, dtype=int)
        for j in range(self.p.n):
            start = j * self.bits_per_job
            bits = chrom[start : start + self.bits_per_job]
            val = int("".join(str(b) for b in bits), 2)
            assignment[j] = val % self.p.m
        return assignment

    def encode(self, assignment):
        """Converts Discrete assignment to binary chromosome."""
        chrom = []
        for val in assignment:
            # Binary string with padding based on bits_per_job
            b_str = format(val, f'0{self.bits_per_job}b')
            chrom.extend([int(b) for b in b_str])
        return np.array(chrom)

    def select(self, pop, fits):
        """Tournament Selection (K=3)."""
        k = 3
        candidates_idx = random.sample(range(len(pop)), k)
        # GAP  is profit maximization hence, max()
        best_idx = max(candidates_idx, key=lambda idx: fits[idx])
        return pop[best_idx]

    def run(self):
        # Initial Population: Random -> Decode -> Repair -> Encode
        pop_list = []
        for _ in range(self.POP):
            chrom = np.random.randint(2, size=self.chrom_length)
            decoded = self.decode(chrom)
            repaired = self.p.repair_assignment(decoded)
            pop_list.append(self.encode(repaired))
        pop = np.array(pop_list)

        best_sol = None
        best_fit = -float('inf')
        history = []
        elite_count = max(1, int(self.ELITE_FRAC * self.POP))

        for t in range(self.GEN):
            # Evaluate current population
            fits = np.array([self.p.evaluate(self.decode(x)) for x in pop])

            # Elitism logic: Descending order for profit
            sort_idx = np.argsort(fits)[::-1]
            elites = pop[sort_idx[:elite_count]].copy()

            if fits[sort_idx[0]] > best_fit:
                best_fit = fits[sort_idx[0]]
                best_sol = self.decode(elites[0]).copy()

            new_pop = list(elites)

            # Adaptive Mutation Rate (Procurement Logic)
            current_mut_rate = self.MUT_MAX - ((self.MUT_MAX - self.MUT_MIN) * (t / self.GEN))

            while len(new_pop) < self.POP:
                p1 = self.select(pop, fits)
                p2 = self.select(pop, fits)

                if random.random() < self.CROSS:
                    # 2-Point Crossover for better structural mixing
                    pt1 = np.random.randint(1, self.chrom_length - 1)
                    pt2 = np.random.randint(pt1 + 1, self.chrom_length)
                    c1 = np.concatenate([p1[:pt1], p2[pt1:pt2], p1[pt2:]])
                    c2 = np.concatenate([p2[:pt1], p1[pt1:pt2], p2[pt2:]])
                else:
                    c1, c2 = p1.copy(), p2.copy()

                for c in [c1, c2]:
                    # Bit-flip Mutation
                    mut_mask = np.random.rand(len(c)) < current_mut_rate
                    c[mut_mask] = 1 - c[mut_mask]

                    # Integrated Repair Step
                    decoded_c = self.decode(c)
                    repaired_c = self.p.repair_assignment(decoded_c)
                    final_chrom = self.encode(repaired_c)
                    
                    new_pop.append(final_chrom)
                    if len(new_pop) >= self.POP: break

            pop = np.array(new_pop[:self.POP])
            history.append(best_fit)

        return best_fit, best_sol, history

In [4]:
# Base Class for Continuous Algorithms
class ContinuousOptimizer:
    def __init__(self, problem, pop_size=100, gens=200):
        self.prob = problem
        self.pop_size = pop_size
        self.gens = gens
        self.dim = problem.n
        self.bounds = (0, problem.m - 1e-5)

    def decode(self, continuous_vector):
        return np.floor(np.clip(continuous_vector, self.bounds[0], self.bounds[1])).astype(int)

In [ ]:
class RCGA:
    def __init__(self, problem, pop_size=100, gens=500):
        self.p = problem
        self.POP = pop_size
        self.GEN = gens
        self.MUT_MAX = 0.20 # In Real coding we use higher mutation
        self.MUT_MIN = 0.05
        self.CROSS = 0.8
        self.ELITE_FRAC = 0.1
        # Range: 0 to m-1 Agents
        self.bounds = (0, self.p.m - 1e-5)

    def decode(self, chrom):
        """Converts continuous values to discrete agent IDs."""
        return np.floor(np.clip(chrom, self.bounds[0], self.bounds[1])).astype(int)

    def select(self, pop, fits):
        """Tournament Selection (K=3)."""
        k = 3
        candidates_idx = random.sample(range(len(pop)), k)
        best_idx = max(candidates_idx, key=lambda idx: fits[idx])
        return pop[best_idx]

    def run(self):
        self.p.clear_caches()
        # 1. Initial Population: Random -> Decode -> Repair -> Repopulate (Continuous)
        # Note: We bring back repaired discrete solutions in continuous space
        pop = []
        for _ in range(self.POP):
            chrom = np.random.uniform(self.bounds[0], self.bounds[1], self.p.n)
            repaired_discrete = self.p.repair_assignment(self.decode(chrom))
            pop.append(repaired_discrete.astype(float)) # Store as float for RCGA
        pop = np.array(pop)

        best_sol = None
        best_fit = -float('inf')
        history = []
        elite_count = max(1, int(self.ELITE_FRAC * self.POP))

        for t in range(self.GEN):
            # Evaluate
            fits = np.array([self.p.evaluate(self.decode(x)) for x in pop])

            # Elitism
            sort_idx = np.argsort(fits)[::-1]
            elites = pop[sort_idx[:elite_count]].copy()

            if fits[sort_idx[0]] > best_fit:
                best_fit = fits[sort_idx[0]]
                best_sol = self.decode(elites[0]).copy()

            new_pop = list(elites)

            # Adaptive Mutation Rate (Decreases over time)
            current_mut_rate = self.MUT_MAX - ((self.MUT_MAX - self.MUT_MIN) * (t / self.GEN))

            while len(new_pop) < self.POP:
                p1 = self.select(pop, fits)
                p2 = self.select(pop, fits)

                if random.random() < self.CROSS:
                    # Arithmetic Crossover
                    alpha = np.random.rand(self.p.n)
                    c1 = alpha * p1 + (1 - alpha) * p2
                    c2 = (1 - alpha) * p1 + alpha * p2
                else:
                    c1, c2 = p1.copy(), p2.copy()

                for c in [c1, c2]:
                    # Gaussian Mutation
                    if random.random() < current_mut_rate:
                        # Every job can get any agent among 0 se m-1
                        mutation_mask = np.random.rand(self.p.n) < 0.1 # 10% genes mutate
                        c[mutation_mask] = np.random.uniform(self.bounds[0], self.bounds[1], np.sum(mutation_mask))

                    # Integrated Repair Step
                    decoded_c = self.decode(c)
                    repaired_discrete = self.p.repair_assignment(decoded_c)
                    new_pop.append(repaired_discrete.astype(float))

                    if len(new_pop) >= self.POP: break

            pop = np.array(new_pop[:self.POP])
            history.append(best_fit)

        return best_fit, best_sol, history

In [6]:
class PSO(ContinuousOptimizer):
    def run(self):
        self.prob.clear_caches()
        # Inertia weight decay
        w_max, w_min = 0.9, 0.4
        c1, c2 = 2.0, 2.0 # Standard coefficients
        pos = np.random.uniform(self.bounds[0], self.bounds[1], (self.pop_size, self.dim))
        vel = np.random.uniform(-1, 1, (self.pop_size, self.dim))
        pbest_pos = pos.copy()
        pbest_fit = np.array([self.prob.evaluate(self.prob.repair_assignment(self.decode(p))) for p in pos])
        gbest_idx = np.argmax(pbest_fit)
        gbest_fit = pbest_fit[gbest_idx]
        gbest_pos = pbest_pos[gbest_idx].copy()
        history = []

        for gen in range(self.gens):
            w = w_max - (gen/self.gens)*(w_max - w_min) # Adaptive Weight
            for i in range(self.pop_size):
                r1, r2 = np.random.rand(self.dim), np.random.rand(self.dim)
                vel[i] = w*vel[i] + c1*r1*(pbest_pos[i]-pos[i]) + c2*r2*(gbest_pos-pos[i])
                pos[i] = np.clip(pos[i] + vel[i], self.bounds[0], self.bounds[1])
                
                curr_fit = self.prob.evaluate(self.prob.repair_assignment(self.decode(pos[i])))
                if curr_fit > pbest_fit[i]:
                    pbest_fit[i], pbest_pos[i] = curr_fit, pos[i].copy()
                    if curr_fit > gbest_fit:
                        gbest_fit, gbest_pos = curr_fit, pos[i].copy()
            history.append(gbest_fit)
        return gbest_fit, self.prob.repair_assignment(self.decode(gbest_pos)), history

In [7]:
class TLBO(ContinuousOptimizer):
    def run(self):
        self.prob.clear_caches()
        pop = np.random.uniform(self.bounds[0], self.bounds[1], (self.pop_size, self.dim))
        fits = np.array([self.prob.evaluate(self.prob.repair_assignment(self.decode(p))) for p in pop])
        history = []
        for gen in range(self.gens):
            # Teacher Phase
            best_idx = np.argmax(fits)
            teacher = pop[best_idx]
            mean_v = np.mean(pop, axis=0)
            for i in range(self.pop_size):
                tf = np.random.randint(1, 3) # Randomly 1 or 2
                new_sol = np.clip(pop[i] + np.random.rand() * (teacher - tf * mean_v), self.bounds[0], self.bounds[1])
                new_fit = self.prob.evaluate(self.prob.repair_assignment(self.decode(new_sol)))
                if new_fit > fits[i]:
                    pop[i], fits[i] = new_sol, new_fit
            
            # Learner Phase
            for i in range(self.pop_size):
                partner = np.random.choice([idx for idx in range(self.pop_size) if idx != i])
                if fits[i] > fits[partner]:
                    diff = pop[i] - pop[partner]
                else:
                    diff = pop[partner] - pop[i]
                new_sol = np.clip(pop[i] + np.random.rand() * diff, self.bounds[0], self.bounds[1])
                new_fit = self.prob.evaluate(self.prob.repair_assignment(self.decode(new_sol)))
                if new_fit > fits[i]:
                    pop[i], fits[i] = new_sol, new_fit
            history.append(np.max(fits))
        return np.max(fits), self.prob.repair_assignment(self.decode(pop[np.argmax(fits)])), history

In [ ]:
class ACO:
    def __init__(self, problem, pop_size=50, gens=200, alpha=1.0, beta=2.5, rho=0.15):
        self.prob = problem
        self.pop_size = pop_size
        self.gens = gens
        self.alpha, self.beta, self.rho = alpha, beta, rho

    def run(self):
        self.prob.clear_caches() # <--- Keeps execution time fair
        eta = self.prob.profits / (self.prob.resources + 1e-6)
        tau = np.ones((self.prob.m, self.prob.n))
        best_fit, best_sol, history = -np.inf, None, []

        for gen in range(self.gens):
            pop_sols, pop_fits = [], []
            for _ in range(self.pop_size):
                sol = np.zeros(self.prob.n, dtype=int)
                for j in range(self.prob.n):
                    attraction = (tau[:, j]**self.alpha) * (eta[:, j]**self.beta)
                    
                    # --- FAST ROULETTE WHEEL ---
                    cumsum_probs = np.cumsum(attraction)
                    cumsum_probs /= cumsum_probs[-1] # Normalize
                    sol[j] = np.searchsorted(cumsum_probs, np.random.rand())
                
                repaired = self.prob.repair_assignment(sol)
                fit = self.prob.evaluate(repaired)
                pop_sols.append(repaired)
                pop_fits.append(fit)
                
                if fit > best_fit:
                    best_fit, best_sol = fit, repaired.copy()
            
            tau *= (1 - self.rho)
            sorted_indices = np.argsort(pop_fits)[::-1]
            top_n = max(1, self.pop_size // 5) 
            for idx in sorted_indices[:top_n]:
                reward = 1.0 / (1 + (best_fit - pop_fits[idx])/best_fit)
                for j in range(self.prob.n):
                    tau[pop_sols[idx][j], j] += reward
                    
            history.append(best_fit)
        return best_fit, best_sol, history

In [9]:
class DE(ContinuousOptimizer):
    def run(self, F=0.8, CR=0.9):
        self.prob.clear_caches()
        pop = np.random.uniform(self.bounds[0], self.bounds[1], (self.pop_size, self.dim))
        fits = np.array([self.prob.evaluate(self.prob.repair_assignment(self.decode(p))) for p in pop])
        history = []
        for gen in range(self.gens):
            for i in range(self.pop_size):
                # Strategy: DE/best/1/bin
                best_vec = pop[np.argmax(fits)]
                r1, r2 = pop[np.random.choice(self.pop_size, 2, replace=False)]
                mutant = np.clip(best_vec + F * (r1 - r2), self.bounds[0], self.bounds[1])
                
                cross_mask = np.random.rand(self.dim) < CR
                trial = np.where(cross_mask, mutant, pop[i])
                
                trial_fit = self.prob.evaluate(self.prob.repair_assignment(self.decode(trial)))
                if trial_fit > fits[i]:
                    pop[i], fits[i] = trial, trial_fit
            history.append(np.max(fits))
        return np.max(fits), self.prob.repair_assignment(self.decode(pop[np.argmax(fits)])), history

In [10]:
class ABC(ContinuousOptimizer):
    def run(self, limit=30):
        self.prob.clear_caches()
        pop = np.random.uniform(self.bounds[0], self.bounds[1], (self.pop_size, self.dim))
        fits = np.array([self.prob.evaluate(self.prob.repair_assignment(self.decode(p))) for p in pop])
        trials = np.zeros(self.pop_size)
        history = []
        
        best_idx = np.argmax(fits)
        best_fit = fits[best_idx]
        best_pos = pop[best_idx].copy()

        for gen in range(self.gens):
            # Employed Bees Phase
            for i in range(self.pop_size):
                k = np.random.choice([idx for idx in range(self.pop_size) if idx != i])
                phi = np.random.uniform(-1, 1, self.dim)
                # Global best guided search
                new_sol = pop[i] + phi * (pop[i] - pop[k]) + np.random.rand()*(best_pos - pop[i])
                new_sol = np.clip(new_sol, self.bounds[0], self.bounds[1])
                
                new_fit = self.prob.evaluate(self.prob.repair_assignment(self.decode(new_sol)))
                if new_fit > fits[i]:
                    pop[i], fits[i], trials[i] = new_sol, new_fit, 0
                else:
                    trials[i] += 1

            # Onlooker Bees Phase
            # Probability proportional to fitness
            prob_dist = (fits - np.min(fits)) / (np.sum(fits - np.min(fits)) + 1e-9)
            for _ in range(self.pop_size):
                i = np.random.choice(self.pop_size, p=prob_dist)
                k = np.random.choice([idx for idx in range(self.pop_size) if idx != i])
                new_sol = np.clip(pop[i] + np.random.uniform(-1,1)*(pop[i]-pop[k]), self.bounds[0], self.bounds[1])
                new_fit = self.prob.evaluate(self.prob.repair_assignment(self.decode(new_sol)))
                if new_fit > fits[i]:
                    pop[i], fits[i], trials[i] = new_sol, new_fit, 0
                else:
                    trials[i] += 1

            # Scout Bees Phase
            for i in range(self.pop_size):
                if trials[i] > limit:
                    pop[i] = np.random.uniform(self.bounds[0], self.bounds[1], self.dim)
                    fits[i] = self.prob.evaluate(self.prob.repair_assignment(self.decode(pop[i])))
                    trials[i] = 0

            if np.max(fits) > best_fit:
                best_fit = np.max(fits)
                best_pos = pop[np.argmax(fits)].copy()
            
            history.append(best_fit)
            
        return best_fit, self.prob.repair_assignment(self.decode(best_pos)), history

In [ ]:
# Data Parser and Execution Engine

def parse_gap_data(raw_data):
    """Parses standard OR-library GAP file formats."""
    lines = raw_data.replace('\n', ' ').split()
    tokens = [int(x) for x in lines if x.strip()]
    
    num_instances = tokens[0] # first token is number of instances
    ptr = 1
    
    m = tokens[ptr]
    n = tokens[ptr+1]
    ptr += 2
    
    profits = np.array(tokens[ptr:ptr+(m*n)]).reshape((m, n))
    ptr += (m*n)
    
    resources = np.array(tokens[ptr:ptr+(m*n)]).reshape((m, n))
    ptr += (m*n)
    
    capacities = np.array(tokens[ptr:ptr+m])
    
    return GAPProblem(m, n, profits, resources, capacities)

In [ ]:
def run_experiment(num_runs = 20):
    # 1. Instances Data
    instances_raw = {
        24: """5 8 24
               25 23 20 16 19 22 20 16 15 22 15 21 20 23 20 22 19 25 25 24 21 17 23 17
               16 19 22 22 19 23 17 24 15 24 18 19 20 24 25 25 19 24 18 21 16 25 15 20
               20 18 23 23 23 17 19 16 24 24 17 23 19 22 23 25 23 18 19 24 20 17 23 23
               16 16 15 23 15 15 25 22 17 20 19 16 17 17 20 17 17 18 16 18 15 25 22 17
               17 23 21 20 24 22 25 17 22 20 16 22 21 23 24 15 22 25 18 19 19 17 22 23
               24 21 23 17 21 19 19 17 18 24 15 15 17 18 15 24 19 21 23 24 17 20 16 21
               18 21 22 23 22 15 18 15 21 22 15 23 21 25 25 23 20 16 25 17 15 15 18 16
               19 24 18 17 21 18 24 25 18 23 21 15 24 23 18 18 23 23 16 20 20 19 25 21
               8 18 22 5 11 11 22 11 17 22 11 20 13 13 7 22 15 22 24 8 8 24 18 8
               24 14 11 15 24 8 10 15 19 25 6 13 10 25 19 24 13 12 5 18 10 24 8 5
               22 22 21 22 13 16 21 5 25 13 12 9 24 6 22 24 11 21 11 14 12 10 20 6
               13 8 19 12 19 18 10 21 5 9 11 9 22 8 12 13 9 25 19 24 22 6 19 14
               25 16 13 5 11 8 7 8 25 20 24 20 11 6 10 10 6 22 10 10 13 21 5 19
               19 19 5 11 22 24 18 11 6 13 24 24 22 6 22 5 14 6 16 11 6 8 18 10
               24 10 9 10 6 15 7 13 20 8 7 9 24 9 21 9 11 19 10 5 23 20 5 21
               6 9 9 5 12 10 16 15 19 18 20 18 16 21 11 12 22 16 21 25 7 14 16 10
               36 35 38 34 32 34 31 34""",
        
        32: """5 8 32
               21 22 17 16 17 24 17 15 25 18 18 21 19 23 24 25 25 16 22 20 20 20 20 21 24 19 22 15 20 17 17 22
               18 24 18 22 19 25 23 15 24 22 19 19 20 25 23 25 20 16 22 24 23 17 24 18 18 22 15 20 17 19 15 22
               24 20 20 15 21 15 21 22 19 16 16 19 21 21 15 17 24 17 19 18 21 25 17 24 21 24 25 17 19 24 21 22
               21 20 16 21 20 18 22 17 18 19 16 15 21 24 23 19 20 20 16 16 21 17 17 20 22 21 16 16 15 17 24 24
               20 19 20 25 17 22 17 18 20 23 20 16 17 24 18 16 20 23 22 15 15 23 18 21 25 20 18 19 24 25 17 16
               17 24 22 23 17 20 19 16 24 21 25 16 25 23 22 23 15 15 15 20 20 21 23 24 21 25 23 22 25 24 21 17
               15 22 23 24 17 23 16 16 18 19 15 15 23 18 24 25 18 25 18 15 16 22 19 18 19 22 18 23 16 15 25 25
               20 18 18 24 25 20 16 19 25 24 25 24 15 22 21 23 15 24 17 15 24 16 25 19 19 22 17 16 15 20 20 18
               12 24 13 15 19 7 6 25 15 19 12 24 24 25 15 19 15 24 8 18 9 23 10 8 11 10 18 21 20 14 20 16
               10 20 10 24 10 25 11 11 23 14 6 16 5 20 15 11 11 19 5 7 25 9 5 24 6 13 13 25 19 18 22 7
               24 6 16 19 18 9 20 23 19 23 15 10 25 25 12 17 16 8 10 9 9 21 19 11 14 22 20 23 23 9 8 25
               17 19 11 25 24 11 16 8 22 18 5 6 21 22 19 6 14 14 13 12 21 20 8 12 25 11 13 20 22 22 15 8
               8 8 19 24 25 20 18 22 9 21 19 18 22 5 6 6 23 15 12 25 14 10 18 14 10 5 22 7 17 16 21 13
               10 23 10 17 19 7 11 22 25 19 12 25 5 20 9 17 11 24 7 13 23 19 25 7 6 13 9 5 15 7 21 7
               6 6 20 5 21 7 13 7 11 7 19 16 15 22 19 13 24 12 8 25 17 12 14 5 11 9 18 5 11 22 14 12
               13 21 7 8 24 15 9 6 16 5 6 10 23 20 8 24 13 9 7 23 6 21 25 9 6 17 21 6 8 17 18 25
               51 45 52 50 49 46 42 44""",

        40: """5 8 40
               22 21 20 16 15 17 18 15 24 16 18 16 23 16 18 15 17 17 23 21 17 19 22 22 18 16 25 18 25 21 23 23 15 19 16 20 15 18 23 23
               20 25 24 17 20 19 23 22 17 16 19 24 18 18 19 24 16 25 20 20 24 22 23 18 17 16 15 23 23 16 15 20 22 22 25 20 24 17 24 24
               22 24 17 20 21 18 16 22 24 22 24 18 16 20 15 15 24 16 21 15 20 18 17 15 21 15 22 18 22 17 24 21 18 16 20 15 16 19 19 21
               19 24 24 19 17 18 25 16 20 23 22 18 16 15 16 21 25 16 25 22 25 15 19 20 16 25 17 23 23 16 25 22 22 22 20 16 22 15 20 20
               22 20 24 21 24 21 24 20 21 25 22 19 17 20 23 23 20 24 23 16 19 23 20 22 19 21 20 22 19 23 21 17 21 22 24 17 22 18 15 21
               20 20 18 20 22 15 20 16 25 22 19 25 17 23 18 22 24 24 15 25 23 18 15 18 17 20 16 20 15 22 18 25 18 17 21 17 24 17 24 22
               25 24 22 25 17 19 23 19 16 15 15 16 21 22 23 15 20 23 22 20 16 24 20 24 25 15 21 15 25 18 16 15 24 16 15 17 17 17 15 15
               21 22 25 18 25 15 24 15 25 24 15 18 16 15 17 18 15 19 22 24 25 21 23 24 17 24 20 23 20 22 24 16 25 25 18 15 24 23 24 22
               9 25 5 12 14 14 19 25 9 21 13 17 19 9 12 10 17 8 13 8 13 7 25 17 7 19 11 6 15 21 6 9 22 9 20 21 7 7 12 18
               23 8 15 23 7 7 10 7 16 25 9 11 23 22 9 12 17 5 21 13 20 25 8 22 9 13 10 7 15 8 19 6 22 14 22 17 8 14 19 23
               20 14 16 12 19 5 20 11 18 14 23 19 8 17 22 13 5 19 15 20 22 20 23 17 11 5 23 12 6 15 14 24 25 17 17 19 18 7 10 19
               17 19 25 20 9 7 20 11 22 23 5 16 9 18 21 24 22 8 13 25 18 10 25 11 9 23 5 17 20 25 22 18 24 17 7 11 5 23 9 14
               19 24 5 8 5 11 8 14 14 14 10 6 10 25 13 14 24 16 8 24 16 19 15 18 13 22 18 21 12 10 13 11 14 7 21 6 11 16 12 24
               18 24 20 21 13 14 13 23 5 22 11 10 5 7 12 7 12 11 25 20 13 11 6 10 5 5 23 22 14 11 10 14 14 24 12 8 9 19 23 25
               16 17 18 14 20 11 25 14 20 5 6 16 8 13 20 10 14 15 14 17 19 17 15 18 22 10 24 23 17 11 18 5 13 21 7 25 14 6 21 6
               7 19 14 25 18 13 18 17 22 10 9 7 13 14 9 25 13 7 8 11 9 6 22 9 14 6 11 16 8 17 5 6 20 6 21 9 21 18 17 12
               55 58 63 64 57 57 60 53""",

        48: """5 8 48
               21 25 18 18 21 25 22 18 18 23 17 24 16 21 24 20 23 24 23 24 19 24 21 20 25 24 22 23 18 25 20 17 17 20 17 17 23 19 18 15 19 24 24 25 16 23 17 18
               19 16 22 23 24 21 18 18 17 17 25 21 24 17 17 22 15 24 18 21 21 23 19 24 19 22 17 18 24 24 21 24 16 21 23 22 19 20 24 20 19 21 24 25 22 15 19 16
               23 19 16 20 20 18 24 15 24 17 24 21 15 19 22 20 20 24 20 16 17 19 18 18 17 24 23 15 16 22 25 24 22 16 25 20 16 20 18 15 24 24 15 24 23 24 18 25
               16 16 21 22 20 22 22 19 25 16 18 19 22 19 18 22 25 20 23 17 18 21 16 20 16 15 21 20 18 16 23 16 22 22 18 22 17 15 15 18 15 20 15 21 19 23 17 25
               16 15 22 23 16 23 20 20 23 16 19 16 18 20 18 22 16 15 22 24 24 21 22 16 19 22 17 20 20 21 15 15 21 16 16 15 15 16 24 25 19 24 22 22 20 19 17 24
               24 19 21 21 25 24 20 25 21 24 18 21 21 15 23 24 18 23 20 25 22 20 17 25 16 22 19 18 22 25 16 16 22 23 22 23 16 21 17 21 20 16 15 23 24 24 18 22
               24 23 18 22 25 18 16 21 24 25 18 16 16 23 25 15 16 19 18 20 19 18 24 18 22 16 25 25 22 18 16 17 15 25 24 23 23 21 25 22 24 19 16 21 16 15 16 25
               15 21 23 24 17 23 23 20 18 17 16 25 16 22 20 17 21 18 17 17 18 24 24 24 24 18 20 21 23 24 19 18 18 19 22 16 23 16 17 22 18 21 24 25 25 22 18 19
               7 6 8 6 13 16 9 14 15 5 10 6 10 7 11 10 13 7 11 11 9 15 10 11 10 14 16 7 12 12 7 11 15 16 11 8 13 10 5 7 6 9 6 5 11 9 13 10
               10 7 5 10 16 14 15 10 16 15 16 16 5 12 16 6 12 12 10 9 15 11 13 8 12 10 13 14 12 13 9 7 8 14 6 14 15 12 10 9 7 7 9 16 6 5 13 13
               16 8 13 16 16 11 5 14 10 5 12 13 12 15 16 5 16 5 15 6 7 9 13 13 11 9 14 13 8 12 5 13 8 9 14 7 13 14 8 9 14 14 12 10 10 5 14 8
               15 7 5 7 14 15 12 6 5 9 13 16 10 6 15 9 15 5 16 6 11 13 15 8 13 9 12 13 14 15 12 14 8 14 6 12 15 8 12 5 10 6 14 7 16 14 10 9
               8 10 9 14 7 10 7 12 10 13 16 5 14 15 7 12 16 13 11 10 5 10 15 10 8 5 8 7 6 6 6 15 7 6 14 12 9 6 6 16 12 15 15 10 8 16 7 5
               13 9 11 13 5 8 13 7 10 8 10 11 5 15 14 6 8 13 8 8 9 12 6 11 15 6 15 7 16 16 13 16 8 15 7 5 15 11 12 5 11 13 12 9 10 7 10 13
               16 15 14 10 8 14 8 11 11 8 11 8 16 8 12 8 16 12 15 12 6 9 11 5 5 8 16 16 7 14 11 7 11 14 15 8 13 14 11 7 12 16 6 12 8 15 11 14
               12 13 7 14 7 10 5 7 8 5 8 9 10 16 6 5 9 5 15 12 7 13 10 16 9 10 16 5 16 11 7 10 16 8 15 6 10 12 12 7 7 13 5 11 15 8 10 6
               48 53 52 52 48 50 53 47"""
    }

    job_sizes = sorted(instances_raw.keys())
    results_avg_fitness = {name: [] for name in ["Binary Coded GA", "Real Coded GA", "PSO", "TLBO", "Differential Evolution", "Ant Colony Opt (ACO)", "Artificial Bee Colony (ABC)"]}
    results_best_fitness = {name: [] for name in results_avg_fitness.keys()} # saving best results for Future use
    results_time = {name: [] for name in results_avg_fitness.keys()}

    for size in job_sizes:
        print(f"--- Running Instance with {size} Jobs ({num_runs} Iterations) ---")
        prob = parse_gap_data(instances_raw[size])
        
        algo_factories = {
            "Binary Coded GA": lambda: BCGA(prob, pop_size=100, gens=200),
            "Real Coded GA": lambda: RCGA(prob, pop_size=100, gens=200),
            "PSO": lambda: PSO(prob, pop_size=100, gens=200),
            "TLBO": lambda: TLBO(prob, pop_size=100, gens=200),
            "Differential Evolution": lambda: DE(prob, pop_size=100, gens=200),
            "Ant Colony Opt (ACO)": lambda: ACO(prob, pop_size=50, gens=200),
            "Artificial Bee Colony (ABC)": lambda: ABC(prob, pop_size=100, gens=200)
        }

        for name, create_algo in algo_factories.items():
            total_fit = 0
            total_time = 0
            overall_best_fit = -float('inf')  # variable to track best profit
            
            for i in range(num_runs):
                algo = create_algo() 
                start = time.perf_counter()
                best_f, _, _ = algo.run()
                end = time.perf_counter()
                
                total_fit += best_f
                total_time += (end - start)
                
                # If current run is better than previous run then save it
                if best_f > overall_best_fit:
                    overall_best_fit = best_f
            
            # Calculate Averages
            avg_fit = total_fit / num_runs
            avg_time = total_time / num_runs
            
            results_avg_fitness[name].append(avg_fit)
            results_best_fitness[name].append(overall_best_fit) 
            results_time[name].append(avg_time)
            
            print(f"[{name}] Avg Profit: {avg_fit:.2f} | Best Profit: {overall_best_fit:.2f} | Avg Time: {avg_time:.2f}s")

    return job_sizes, results_avg_fitness, results_best_fitness, results_time

In [ ]:
# 2. Execution
job_sizes, fitness_data, best_fitness, time_data = run_experiment()

In [ ]:
def plot_grouped_bars(x_labels, data_dict, title, ylabel, is_fitness=True):
    n_groups = len(x_labels)
    n_algos = len(data_dict)
    fig, ax = plt.subplots(figsize=(16, 8))
    
    index = np.arange(n_groups)
    
    bar_width = 0.12 
    opacity = 0.85
    
    colors = plt.cm.get_cmap('tab10', n_algos).colors
    
    for i, (algo_name, values) in enumerate(data_dict.items()):
        # i * bar_width ensures bars are adjacent
        bars = ax.bar(index + i * bar_width, values, bar_width, 
                      alpha=opacity, label=algo_name, color=colors[i], edgecolor='black', linewidth=0.5)
        
        for bar in bars:
            height = bar.get_height()
            offset = height * 0.015
            
            # Vertical labels above bars (Showing averages)
            ax.text(bar.get_x() + bar.get_width()/2., height + offset,
                    f'{height:.1f}' if is_fitness else f'{height:.2f}s',
                    ha='center', va='bottom', rotation='vertical', 
                    fontsize=8.5, fontweight='bold', color='black')

    ax.set_xlabel('Number of Jobs', fontsize=12, fontweight='bold')
    ax.set_ylabel(ylabel, fontsize=12, fontweight='bold')
    ax.set_title(title, fontsize=16, fontweight='bold', pad=30)
    
    # Aligns X-axis ticks in the center of cluster ke center mein alignment
    ax.set_xticks(index + bar_width * (n_algos - 1) / 2)
    # Reverted to automatic labels
    ax.set_xticklabels(x_labels)
    
    # moved the legend outside
    ax.legend(loc='upper left', bbox_to_anchor=(1, 1), title="Algorithms", frameon=True)
    
    # Y-axis limit stretch for vertical text
    ax.set_ylim(0, ax.get_ylim()[1] * 1.2) 
    
    ax.grid(axis='y', linestyle=':', alpha=0.5)
    plt.tight_layout()
    plt.show()

In [ ]:
plot_grouped_bars(job_sizes, fitness_data, 'Average Fitness Comparison (20 Runs)', 'Average Fitness (Profit)', is_fitness=True)
plot_grouped_bars(job_sizes, time_data, 'Average Execution Time Comparison (20 Runs)', 'Average Time (Seconds)', is_fitness=False)